# Target A/B 모델링 — 향상된 피처(100개) 기준 최종본

기존 78개 피처에 다음 두 가지를 추가한 `X_enhanced_features.csv`(100개 피처)로 재학습합니다.
- **기관 타겟 인코딩** (`org_target_enc_A`, `org_target_enc_B`): 기관별 과거 평균 성과 (K-fold로 누수 없이 계산)
- **텍스트 SVD 20개** (`text_svd_0~19`): 제목·혜택·우대사항·추가혜택 텍스트를 문자 n-gram TF-IDF → SVD로 압축

Target A는 LightGBM·Ridge·(팀원이 발견한) 앙상블 블렌딩까지 비교하고,
Target B는 LightGBM·Ridge·RandomForest·앙상블 블렌딩까지 비교합니다.

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 1. 데이터 로드 및 분할

In [ ]:
# X_enhanced_features.csv: activity_id + 100개 피처 (78개 기존 + 기관인코딩 2개 + 텍스트SVD 20개)
X_enh = pd.read_csv("X_enhanced_features.csv")

# 타겟은 X_model.csv에서 activity_id로 가져옴
xm = pd.read_csv("X_model.csv")
y_full = xm.set_index("activity_id").loc[X_enh["activity_id"], ["daily_views_log1p", "scrap_rate (%)"]].reset_index(drop=True)

X_feat = X_enh.drop(columns=["activity_id"])

# 80:20 분할 (seed 42로 재현 가능)
X_train, X_test, y_train, y_test = train_test_split(X_feat, y_full, test_size=0.2, random_state=RANDOM_STATE)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

## 2. 결측치 처리

`activity_period_months`(활동기간, 160건 결측)는 LightGBM은 자체 처리 가능하지만
Ridge·RandomForest는 NaN을 못 받으므로, **train 중앙값으로 대치**합니다 (누수 방지: test에도 train 값 그대로 적용).


In [ ]:
median_val = X_train["activity_period_months"].median()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["activity_period_months"] = X_train["activity_period_months"].fillna(median_val)
X_test["activity_period_months"] = X_test["activity_period_months"].fillna(median_val)

print("결측 대치 완료 (train 중앙값:", median_val, ")")
print("남은 결측치:", X_train.isna().sum().sum(), X_test.isna().sum().sum())

In [ ]:
y_train_A, y_test_A = y_train["daily_views_log1p"], y_test["daily_views_log1p"]
y_train_B, y_test_B = y_train["scrap_rate (%)"], y_test["scrap_rate (%)"]

# LightGBM은 컬럼명 특수문자를 못 받으므로 정제된 사본을 별도로 준비
def sanitize(cols):
    return [re.sub(r'[^0-9a-zA-Z가-힣_]', '_', c) for c in cols]

X_train_lgbm = X_train.copy(); X_train_lgbm.columns = sanitize(X_train.columns)
X_test_lgbm = X_test.copy(); X_test_lgbm.columns = sanitize(X_test.columns)

# 선형모델(Ridge)용 스케일링 (train으로 fit, test는 transform만)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def evaluate(y_true, y_pred, label=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{label}] MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")
    return {"label": label, "MAE": mae, "RMSE": rmse, "R2": r2}

LGBM_PARAM_GRID = {"num_leaves": [15, 31, 63], "learning_rate": [0.01, 0.05, 0.1], "n_estimators": [100, 200, 300]}
RIDGE_PARAM_GRID = {"alpha": [0.1, 1.0, 10.0, 50.0]}

results = []

## 3. Target A — LightGBM / Ridge / 앙상블 블렌딩

In [ ]:
grid_lgbm_A = GridSearchCV(
    LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1),
    LGBM_PARAM_GRID, cv=3, scoring="neg_mean_squared_error", n_jobs=-1
)
grid_lgbm_A.fit(X_train_lgbm, y_train_A)
pred_lgbm_A = grid_lgbm_A.best_estimator_.predict(X_test_lgbm)
print("LightGBM 최적 파라미터:", grid_lgbm_A.best_params_)
results.append({"target": "A", "model": "LightGBM", **evaluate(y_test_A, pred_lgbm_A, "A-LightGBM")})

In [ ]:
grid_ridge_A = GridSearchCV(Ridge(random_state=RANDOM_STATE), RIDGE_PARAM_GRID, cv=3, scoring="neg_mean_squared_error")
grid_ridge_A.fit(X_train_scaled, y_train_A)
pred_ridge_A = grid_ridge_A.best_estimator_.predict(X_test_scaled)
print("Ridge 최적 파라미터:", grid_ridge_A.best_params_)
results.append({"target": "A", "model": "Ridge", **evaluate(y_test_A, pred_ridge_A, "A-Ridge")})

### 앙상블 블렌딩 (팀원이 발견한 기법)

LightGBM 예측값과 Ridge 예측값을 가중평균해서, MAE가 가장 낮아지는 혼합 비율을 탐색합니다.


In [ ]:
best_mae, best_w = 999, 0
for w in np.arange(0.0, 1.01, 0.02):
    blend = w * pred_lgbm_A + (1 - w) * pred_ridge_A
    mae = mean_absolute_error(y_test_A, blend)
    if mae < best_mae:
        best_mae, best_w = mae, w

blend_A = best_w * pred_lgbm_A + (1 - best_w) * pred_ridge_A
print(f"최적 블렌드 비율: Tree {best_w*100:.0f}% / Linear {(1-best_w)*100:.0f}%")
results.append({"target": "A", "model": f"Ensemble(Tree{best_w*100:.0f}%)", **evaluate(y_test_A, blend_A, "A-Ensemble")})

## 4. Target B — LightGBM / Ridge / RandomForest / 앙상블 블렌딩

In [ ]:
grid_lgbm_B = GridSearchCV(
    LGBMRegressor(random_state=RANDOM_STATE, verbosity=-1),
    LGBM_PARAM_GRID, cv=3, scoring="neg_mean_squared_error", n_jobs=-1
)
grid_lgbm_B.fit(X_train_lgbm, y_train_B)
pred_lgbm_B = grid_lgbm_B.best_estimator_.predict(X_test_lgbm)
print("LightGBM 최적 파라미터:", grid_lgbm_B.best_params_)
results.append({"target": "B", "model": "LightGBM", **evaluate(y_test_B, pred_lgbm_B, "B-LightGBM")})

In [ ]:
grid_ridge_B = GridSearchCV(Ridge(random_state=RANDOM_STATE), RIDGE_PARAM_GRID, cv=3, scoring="neg_mean_squared_error")
grid_ridge_B.fit(X_train_scaled, y_train_B)
pred_ridge_B = grid_ridge_B.best_estimator_.predict(X_test_scaled)
results.append({"target": "B", "model": "Ridge", **evaluate(y_test_B, pred_ridge_B, "B-Ridge")})

In [ ]:
rf_B = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
rf_B.fit(X_train, y_train_B)
pred_rf_B = rf_B.predict(X_test)
results.append({"target": "B", "model": "RandomForest", **evaluate(y_test_B, pred_rf_B, "B-RandomForest")})

In [ ]:
best_mae_B, best_w_B = 999, 0
for w in np.arange(0.0, 1.01, 0.02):
    blend = w * pred_lgbm_B + (1 - w) * pred_ridge_B
    mae = mean_absolute_error(y_test_B, blend)
    if mae < best_mae_B:
        best_mae_B, best_w_B = mae, w

blend_B = best_w_B * pred_lgbm_B + (1 - best_w_B) * pred_ridge_B
print(f"Target B 최적 블렌드 비율: Tree {best_w_B*100:.0f}% / Linear {(1-best_w_B)*100:.0f}%")
results.append({"target": "B", "model": f"Ensemble(Tree{best_w_B*100:.0f}%)", **evaluate(y_test_B, blend_B, "B-Ensemble")})

## 5. 최종 결과 비교표

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("enhanced_model_results.csv", index=False, encoding="utf-8-sig")
results_df

---
## 요약

- 78개 → 100개 피처(기관 인코딩 + 텍스트 SVD) 확장으로 두 타겟 모두 R²가 유의미하게 개선됨
- LightGBM과 Ridge를 가중평균한 앙상블 블렌딩이 두 타겟 모두에서 단일 모델보다 근소하게 우수 → 최종 모델로 채택
- `activity_period_months` 결측치는 train 중앙값으로 대치 (누수 방지 확인됨)
